# How Fox frames each player

Regenerates the four framing CSVs behind the "How Fox frames each player" chart:
`politicization_people.csv`, `appearance_talk.csv`, `celebrity_talk.csv`, and `agency.csv`
in `src/data/`. Reads the Fox WNBA headlines from `data/fox_wnba.csv` and the cast of
characters from `src/data/headline_network.json`.

Needs spaCy on top of pandas: `pip install spacy` then `python -m spacy download en_core_web_sm`.


In [ ]:
import csv, json, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

DATA = Path("../data")
OUT = Path("../src/data")

fox = pd.read_csv(DATA / "fox_wnba.csv")
headlines = (fox.assign(headline=fox.headline.str.strip())
                .dropna(subset=["headline"])
                .drop_duplicates("headline"))
headlines = [{"t": r.headline, "d": str(r.date)[:10]} for r in headlines.itertuples()]

# every person the piece tracks, with their role (players / politics / media / league)
nodes = json.loads((OUT / "headline_network.json").read_text())["nodes"]
groups = {n["name"]: n["group"] for n in nodes}
name_rx = {name: re.compile(r"\b" + re.escape(name.split()[-1]) + r"\b", re.I) for name in groups}

totals = Counter()
for row in headlines:
    for name, rx in name_rx.items():
        if rx.search(row["t"]):
            totals[name] += 1

MIN_HEADLINES = 12
print(f"{len(headlines):,} unique headlines · {len(groups)} people tracked")

## Politicization

How often a headline naming this person also carries political language — including the
euphemistic register ("save women's sports", "fairness") that carries the framing without
obvious political words. A politician's name alone doesn't count: during Griner's detainment,
Biden was simply who was in the story.


In [ ]:
TERM_GROUPS = {
    "partisan": r"trump|biden|kamala|harris|maga|republican|democrat|gop|conservative|"
                r"liberal|left-wing|right-wing|woke|congress|senator|governor|"
                r"white house|election|administration|attorney general|politic",
    "womens_sports": r"women'?s sports|girls'? sports|female athlete|biological (?:male|men|women|female)|"
                     r"men (?:don'?t belong|in women)|fairness in|save women|protect(?:ing)? (?:girls|women)|"
                     r"title ix|trans|transgender|gender",
    "activism": r"activis|protest|counterprotest|rally|rallies|boycott|movement|anthem|kneel|"
                r"speak out|speaks out|takes a stand|puts? career on line|petition|march|"
                r"police shooting|mass shooting|church shooting|shooting of|shooting victim|"
                r"shooting death|(?:minneapolis|kenosha|border patrol|ice) shooting|"
                r"shot (?:and killed|dead)|gun (?:control|laws|violence)",
    "race": r"\bracis|\brace\b|white privilege|black lives|\bblm\b|racial|"
            r"\b(?:black|white) (?:wom[ae]n|player|athlete|star)",
    "lgbtq": r"lgbtq|\bgay\b|lesbian|queer|pride|same-sex",
    "patriotism": r"patriot|american flag|god bless|christian|anti-american|un-?american",
    "combatants": r"riley gaines|j\.?k\.? rowling|jk rowling|charlie kirk|enes kanter|"
                  r"dave portnoy|clay travis|jemele hill|megan rapinoe|sarah spain|"
                  r"michele tafoya|outkick|media bias|smear",
}
group_rx = {name: re.compile(p, re.I) for name, p in TERM_GROUPS.items()}

political = Counter()
group_hits = defaultdict(Counter)
examples = defaultdict(list)
for row in headlines:
    matched = [g for g, rx in group_rx.items() if rx.search(row["t"])]
    # a politician's name alone is not political framing
    matched = matched if any(g != "partisan" for g in matched) else []
    for person, rx in name_rx.items():
        if not rx.search(row["t"]):
            continue
        if matched:
            political[person] += 1
            for g in matched:
                group_hits[person][g] += 1
            if len(examples[person]) < 3:
                examples[person].append(row["t"])

people = sorted(
    ({"name": p, "group": groups[p], "n": totals[p], "political": political[p],
      "share": round(political[p] / totals[p], 4),
      "top_groups": "; ".join(g for g, _ in group_hits[p].most_common(3)),
      "examples": examples[p]}
     for p in totals if totals[p] >= MIN_HEADLINES),
    key=lambda r: -r["share"])

rows = [{**{k: v for k, v in p.items() if k != "examples"}, "example": ex}
        for p in people for ex in (p["examples"] or [""])]
pd.DataFrame(rows).to_csv(OUT / "politicization_people.csv", index=False)
pd.DataFrame(people).drop(columns="examples").head(10)

## Appearance & celebrity treatment

How much of a player's coverage is about her clothes (appearance) or her private life plus
her clothes (celebrity). Naming her in a headline containing "outfit" isn't enough — the outfit
is often somebody else's — so spaCy's parse ties the word to her grammatically: she has to be
its possessor ("Cunningham's outfit") or the subject of its verb ("Cunningham wears…").
When the owner is a pronoun ("…confirms she's dating…"), it resolves to the nearest name before it.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

CLOTHING_NOUNS = {"outfit", "fit", "attire", "ensemble", "dress", "bikini",
                  "swimsuit", "lingerie", "photoshoot", "clothes"}
CLOTHING_VERBS = {"wear", "wears", "wearing", "wore", "worn"}
MENTIONS_CLOTHING = re.compile(
    r"\b(outfit|fit check|fit criticism|attire|ensemble|dress(?:es)?|"
    r"bikini|swimsuit|lingerie|photo ?shoot|clothes|"
    r"wear(?:s|ing)?|wore|worn)\b", re.I)

CELEBRITY_NOUNS = {
    "boyfriend", "girlfriend", "husband", "wife", "fiance", "fiancé", "fiancée",
    "partner", "beau", "romance", "relationship", "engagement", "wedding",
    "marriage", "honeymoon", "breakup", "split", "divorce", "ex",
    "baby", "pregnancy", "family", "mom", "mother", "dad", "father", "parents",
    "sister", "brother", "twin", "boo",
    "mansion", "gala", "carpet", "vacation", "birthday", "instagram",
}
CELEBRITY_VERBS = {"date", "dates", "dating", "dated", "marry", "marries",
                   "married", "engaged", "split", "splits", "divorce",
                   "divorced", "expecting", "pregnant"}
MENTIONS_CELEBRITY = re.compile(
    r"\b(boyfriend|girlfriend|husband|wife|fianc[eé]{1,2}|partner|beau|"
    r"romance|relationship|engagement|engaged|wedding|marriage|married|marry|"
    r"honeymoon|break-?up|split|divorce|dating|date night|"
    r"bab(?:y|ies)|pregnan\w+|expecting|famil(?:y|ies)|mom|mother|dad|father|"
    r"parents|sister|brother|twin|"
    r"mansion|net worth|red carpet|gala|"
    r"vacation|birthday|instagram)\b", re.I)
# phrases that live in both registers: only the tabloid sense should score
NOT_CELEBRITY = re.compile(
    r"\b(speaking engagement|fan engagement|"
    r"(?:release|launch|debut|start|due|draft|trade) date|"
    r"home (?:opener|game|court|stand|run)|"
    r"free agency|trade deadline)\b", re.I)

SUBJECT_DEPS = {"nsubj", "nsubjpass", "csubj", "agent"}
POSSESSIVE_DEPS = {"poss", "compound", "nmod", "amod"}
PRONOUNS = {"she", "her", "hers", "herself", "they", "them", "their", "theirs", "he", "his", "him"}

def owner_tokens(token, doc):
    """Which tokens own this clothing/tabloid word — whose outfit is it?"""
    owners = [c for c in token.children if c.dep_ in POSSESSIVE_DEPS]
    node = token
    for _ in range(5):
        head = node.head
        if head is node:
            break
        if head.pos_ in {"VERB", "AUX"}:
            subjects = [c for c in head.children if c.dep_ in SUBJECT_DEPS]
            if subjects:
                owners += subjects
                break
        node = head
    if token.pos_ in {"VERB", "AUX"}:
        owners += [c for c in token.children if c.dep_ in SUBJECT_DEPS]
    if not owners:
        root = next((t for t in doc if t.dep_ == "ROOT"), None)
        if root is not None:
            owners += [c for c in root.children if c.dep_ in SUBJECT_DEPS]
    if not owners:
        owners += list(doc.noun_chunks)[:1]
    return owners

def nearest_named(headline, before):
    """The roster name closest in front of `before` — who a pronoun refers to."""
    best, where = None, -1
    lower = headline.lower()
    for name in groups:
        at = lower.rfind(name.split()[-1].lower(), 0, before)
        if at != -1 and at > where:
            best, where = name, at
    return best

def words_under(tokens):
    return {sub.text.lower().strip(",'’s") for t in tokens for sub in t.subtree}

def attribute(nouns, verbs, prefilter, scrub=None):
    """Set of players each qualifying headline is grammatically about."""
    text_for = (lambda r: scrub.sub(" ", r["t"])) if scrub else (lambda r: r["t"])
    candidates = [r for r in headlines if prefilter.search(text_for(r))]
    attributed = defaultdict(list)
    for row, doc in zip(candidates, nlp.pipe([text_for(r) for r in candidates], batch_size=32)):
        owners_found, pronoun_at = set(), None
        for token in doc:
            word = token.text.lower().strip(",'’s")
            if word not in nouns and word not in verbs:
                continue
            owned = words_under(owner_tokens(token, doc))
            for name in groups:
                if name.split()[-1].lower() in owned:
                    owners_found.add(name)
            if owned & PRONOUNS:
                pronoun_at = min(pronoun_at, token.idx) if pronoun_at is not None else token.idx
        if not owners_found and pronoun_at is not None:
            who = nearest_named(row["t"], pronoun_at)
            if who:
                owners_found.add(who)
        for name in owners_found:
            attributed[name].append(row)
    return attributed

appearance_hits = attribute(CLOTHING_NOUNS, CLOTHING_VERBS, MENTIONS_CLOTHING)
celebrity_hits = attribute(CELEBRITY_NOUNS, CELEBRITY_VERBS, MENTIONS_CELEBRITY, scrub=NOT_CELEBRITY)
print(f"{sum(len(v) for v in appearance_hits.values())} appearance attributions · "
      f"{sum(len(v) for v in celebrity_hits.values())} celebrity")

In [ ]:
def people_table(hits, example_rows):
    people = sorted(
        ({"name": p, "n": totals[p], "hits": len(hits.get(p, [])),
          "share": round(len(hits.get(p, [])) / totals[p], 4),
          "examples": example_rows(p)}
         for p in totals if totals[p] >= MIN_HEADLINES and groups[p] == "players"),
        key=lambda r: -r["share"])
    return people

def long_rows(people):
    return [{**{k: v for k, v in p.items() if k != "examples"},
             "example_date": ex.get("d", ""), "example_title": ex.get("t", "")}
            for p in people for ex in (p["examples"] or [{}])]

appearance_people = people_table(appearance_hits, lambda p: appearance_hits.get(p, [])[:4])
pd.DataFrame(long_rows(appearance_people)).to_csv(OUT / "appearance_talk.csv", index=False)

# celebrity treatment = the private life AND the outfit: three pieces on a divorce
# and one on a pregame fit are the same editorial move twice
dates = {r["t"]: r["d"] for r in headlines}
def combined_examples(p):
    both = {r["t"] for r in celebrity_hits.get(p, [])} | {r["t"] for r in appearance_hits.get(p, [])}
    return [{"d": dates[t], "t": t} for t in sorted(both, key=lambda t: dates[t], reverse=True)][:4]

merged_hits = {p: [{"t": t} for t in ({r["t"] for r in celebrity_hits.get(p, [])}
                                     | {r["t"] for r in appearance_hits.get(p, [])})]
               for p in set(celebrity_hits) | set(appearance_hits)}
celebrity_people = people_table(merged_hits, combined_examples)
pd.DataFrame(long_rows(celebrity_people)).to_csv(OUT / "celebrity_talk.csv", index=False)

pd.DataFrame(celebrity_people).drop(columns="examples").head(10)

## Agency & victim framing

Who acts and who is acted upon. "Alyssa Thomas punches Caitlin Clark" and "Caitlin Clark
punched in the throat" carry the same tone but opposite framing. AGENCY is how often she's the
subject of a verb versus its object (+1 always acting, −1 always acted upon), from the parse.
VICTIM counts any headline where contact language lands on her, however the sentence is built —
the parse is too brittle on headlines to be the gate there.


In [ ]:
SUBJ = {"nsubj", "csubj", "agent"}
PASSIVE = {"nsubjpass"}
OBJ = {"dobj", "obj", "iobj", "dative", "oprd"}
CONTACT_VERBS = {"foul", "shove", "elbow", "punch", "smack", "hit", "strike", "swipe",
                 "hack", "poke", "shoulder", "tackle", "trip", "push", "bump", "injure",
                 "hurt", "bloody", "target", "assault", "attack", "slam", "knock"}
CONTACT_NOUNS = {"foul", "flagrant", "elbow", "punch", "shove", "jab", "smack", "blow",
                 "hit", "contusion", "collision", "takedown", "throat", "eye"}
# reporting verbs say nothing about agency
REPORTING = {"say", "tell", "call", "reveal", "address", "deny", "claim", "announce",
             "ask", "respond", "react", "comment", "add", "note", "explain", "admit",
             "insist", "argue", "state", "speak", "talk", "discuss", "share", "post",
             "write", "report", "confirm", "suggest"}
MIN_PLACED = 12

nlp_full = spacy.load("en_core_web_sm", disable=["ner"])
players = {name for name, g in groups.items() if g == "players"}
surname = {name: name.split()[-1].lower() for name in players}

placed = defaultdict(lambda: {"agent": 0, "patient": 0, "contact_by": 0, "mentions": 0,
                              "as_agent": Counter(), "as_patient": Counter()})
texts = [r["t"] for r in headlines]
for headline, doc in zip(texts, nlp_full.pipe(texts, batch_size=64)):
    lower = headline.lower()
    for name, last in surname.items():
        if last not in lower:
            continue
        stats = placed[name]
        stats["mentions"] += 1
        token = next((t for t in doc if t.text.lower().strip(",'’s") == last), None)
        if token is None:
            continue
        node, verb = token, None
        for _ in range(4):
            head = node.head
            if head is node:
                break
            if head.pos_ in {"VERB", "AUX"}:
                verb = head
                break
            node = head
        if verb is None or verb.lemma_.lower() in REPORTING:
            continue
        lemma = verb.lemma_.lower()
        physical = lemma in CONTACT_VERBS or any(
            c.text.lower() in CONTACT_NOUNS for c in verb.children)
        if token.dep_ in SUBJ:
            stats["agent"] += 1
            stats["as_agent"][lemma] += 1
            if physical:
                stats["contact_by"] += 1
        elif token.dep_ in OBJ or token.dep_ in PASSIVE:
            stats["patient"] += 1
            stats["as_patient"][lemma] += 1

# victim framing: contact language in the headline and she isn't the one delivering it
contact_words = re.compile(
    r"\b(flagrant|foul(?:ed|s)?|shove[ds]?|elbow(?:ed)?|punch(?:ed|es)?|smack(?:ed)?|"
    r"cheap shot|dirty|throat|black eye|eye poke|poked|bloody|takedown|hack(?:ed)?|"
    r"swipe[ds]?|jab|hit|targeted|targeting|injur\w*|hurt)\b", re.I)
by_her = {name: re.compile(
    rf"\b{re.escape(surname[name])}\b[^.]{{0,24}}\b(fouls?|fouled|shoves?|shoved|elbows?|"
    rf"punch(?:es|ed)?|smacks?|smacked|hits?|targets?|hacks?|jabs?)\b", re.I)
    for name in players}

victim = Counter()
victim_examples = defaultdict(list)
for headline in texts:
    if not contact_words.search(headline):
        continue
    for name in players:
        if surname[name] not in headline.lower():
            continue
        if by_her[name].search(headline):
            continue
        victim[name] += 1
        if len(victim_examples[name]) < 4:
            victim_examples[name].append(headline)

people = sorted(
    ({"name": name, "mentions": s["mentions"], "placed": s["agent"] + s["patient"],
      "coverage": round((s["agent"] + s["patient"]) / s["mentions"], 3),
      "agency": round((s["agent"] - s["patient"]) / (s["agent"] + s["patient"]), 3),
      "victim": victim[name], "victim_rate": round(victim[name] / s["mentions"], 4),
      "contact_by": s["contact_by"],
      "top_as_agent": "; ".join(w for w, _ in s["as_agent"].most_common(5)),
      "top_as_patient": "; ".join(w for w, _ in s["as_patient"].most_common(5)),
      "examples": victim_examples[name]}
     for name, s in placed.items() if s["agent"] + s["patient"] >= MIN_PLACED),
    key=lambda r: r["agency"])

rows = [{**{k: v for k, v in p.items() if k != "examples"}, "example": ex}
        for p in people for ex in (p["examples"] or [""])]
pd.DataFrame(rows).to_csv(OUT / "agency.csv", index=False)
pd.DataFrame(people).drop(columns="examples")